<a href="https://colab.research.google.com/github/PrathamVashisht26/Real-Time-Social-Media-Sentiment-Topic-Analysis-Engine-/blob/main/Social_media_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Real-Time Social Media Sentiment & Topic Analysis Engine

A complete NLP pipeline that ingests social media posts, cleans the text with advanced preprocessing, and uses Google Gemini to extract **sentiment** (Positive / Negative / Neutral) and **topic** from each post.

---

## Pipeline Overview

```
┌─────────────────────┐     ┌──────────────────────┐     ┌─────────────────────┐     ┌─────────────────────┐
│  STEP 1             │     │  STEP 2               │     │  STEP 3             │     │  STEP 4             │
│  Data Ingestion     │────▶│  Text Preprocessing  │────▶│  AI Analysis        │────▶│  Visualization      │
│  Twitter API v2     │     │  (NLTK + Regex +      │     │  (Gemini 1.5 Flash) │     │  (Matplotlib)       │
│  or Sample Dataset  │     │   Lemmatization)      │     │  Sentiment + Topic  │     │  Charts & Insights  │
└─────────────────────┘     └──────────────────────┘     └─────────────────────┘     └─────────────────────┘
```

## How to Run

1. **With your own Twitter API key** – Add `BEARER_TOKEN` to Colab Secrets. The notebook will live-fetch tweets.
2. **Without Twitter API** – The notebook automatically falls back to the included `data/sample_tweets.csv` (50 EV tweets). You can still run every other step.
3. **For Gemini analysis** – Add `GEMINI_API_KEY` to Colab Secrets. Get a free key at [aistudio.google.com](https://aistudio.google.com/app/apikey).

> **Note:** Twitter API `search_recent_tweets` requires a paid Basic tier subscription ($100/month). The sample dataset is provided so this project is fully demonstrable without any paid APIs.

---
## Step 1 — Install Dependencies

In [ ]:
# Install all required libraries
!pip install -q tweepy python-dotenv emoji nltk google-generativeai matplotlib seaborn
print("✅ All dependencies installed successfully.")

---
## Step 2 — Data Ingestion

Attempts to fetch live tweets via the Twitter API v2.
If the API is unavailable (rate-limited, no key, or free-tier restriction), **automatically falls back to the included sample dataset** so the rest of the pipeline always runs.

In [ ]:
import tweepy
import pandas as pd
import os

SEARCH_QUERY = '"Electric vehicles" -is:retweet lang:en'
MAX_RESULTS = 100
RAW_OUTPUT_FILE = "raw_tweets.csv"
SAMPLE_DATA_PATH = "data/sample_tweets.csv"  # fallback dataset

df = None  # will hold our data

# ── Try live Twitter API ──────────────────────────────────────────────────────
try:
    from google.colab import userdata
    bearer_token = userdata.get("BEARER_TOKEN")
except Exception:
    bearer_token = os.getenv("BEARER_TOKEN")  # local .env fallback

if bearer_token:
    print("🔑 Bearer token found. Attempting to connect to Twitter API v2...")
    try:
        client = tweepy.Client(bearer_token)
        print("✅ Authentication successful!")
        print(f"   Searching for: {SEARCH_QUERY}")

        response = client.search_recent_tweets(
            query=SEARCH_QUERY,
            max_results=MAX_RESULTS,
            tweet_fields=["created_at", "text", "lang"]
        )

        tweets = response.data
        if not tweets:
            print("⚠️  No tweets returned for this query. Falling back to sample data.")
        else:
            tweet_data = [
                {"id": tweet.id, "created_at": tweet.created_at, "text": tweet.text}
                for tweet in tweets
            ]
            df = pd.DataFrame(tweet_data)
            df.to_csv(RAW_OUTPUT_FILE, index=False)
            print(f"✅ Fetched {len(df)} live tweets and saved to '{RAW_OUTPUT_FILE}'.")
            print(df[["created_at", "text"]].head(3).to_string())

    except tweepy.errors.TooManyRequests:
        print("⚠️  Twitter API rate limit hit (429). This endpoint requires a paid Basic tier subscription.")
        print("   → Falling back to sample dataset.")
    except tweepy.errors.Forbidden:
        print("⚠️  Twitter API access forbidden (403). Your plan may not include search access.")
        print("   → Falling back to sample dataset.")
    except Exception as e:
        print(f"⚠️  Twitter API error: {e}")
        print("   → Falling back to sample dataset.")
else:
    print("ℹ️  No BEARER_TOKEN found in Colab Secrets.")
    print("   → Falling back to sample dataset.")

# ── Fallback: load sample dataset ────────────────────────────────────────────
if df is None:
    try:
        df = pd.read_csv(SAMPLE_DATA_PATH)
        print(f"\n✅ Sample dataset loaded: {len(df)} tweets from '{SAMPLE_DATA_PATH}'.")
        print(df[["created_at", "text"]].head(3).to_string())
    except FileNotFoundError:
        raise FileNotFoundError(
            f"Sample file '{SAMPLE_DATA_PATH}' not found. "
            "Make sure you cloned the full repository including the data/ folder."
        )

print(f"\n📊 Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")

---
## Step 3 — Advanced Text Preprocessing

Cleans raw tweet text through a multi-stage pipeline:
1. **Emoji demojization** — converts `😊` → `:smiling_face:` (preserves sentiment signal)
2. **URL removal** — strips `https://...` and `www....` links
3. **Mention & hashtag stripping** — removes `@user` and `#` symbols
4. **Special character removal** — keeps only alphabetic characters
5. **Stopword removal** — drops common words ("the", "is", "at", ...)
6. **Lemmatization** — reduces words to base form ("running" → "run")

In [ ]:
import re
import emoji
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("✅ NLTK resources ready.")

# ── Preprocessing function ────────────────────────────────────────────────────
def advanced_clean_text(text: str) -> str:
    """Apply a full NLP preprocessing pipeline to a raw tweet string.

    Steps:
        1. Demojize emojis (preserves sentiment signal as text)
        2. Remove URLs
        3. Remove @mentions and strip # from hashtags
        4. Replace underscores with spaces (emoji text connector)
        5. Remove all non-alphabetic characters
        6. Lowercase, tokenize, remove stopwords, lemmatize

    Returns:
        Cleaned and normalised string.
    """
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))

    # 1. Convert emojis to text tokens
    text = emoji.demojize(text)

    # 2. Remove URLs (http:// and www.)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 3. Remove @mentions; remove # symbol but keep hashtag word
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)

    # 4. Replace underscores (from demojized emoji) with spaces
    text = re.sub(r'_', ' ', text)

    # 5. Keep only alphabetic characters and spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 6. Tokenize → lowercase → remove stopwords → lemmatize
    tokens = text.lower().split()
    cleaned_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    return " ".join(cleaned_tokens)


# ── Apply to dataset ──────────────────────────────────────────────────────────
print("\n🔄 Applying advanced text cleaning pipeline...")
df['cleaned_text'] = df['text'].apply(advanced_clean_text)

# Save cleaned data
df.to_csv("cleaned_tweets.csv", index=False)
print("✅ Cleaning complete. Saved to 'cleaned_tweets.csv'.")

# Show comparison
print("\n--- Original vs. Cleaned Text (first 3 rows) ---")
for i, row in df[['text', 'cleaned_text']].head(3).iterrows():
    print(f"\n[{i+1}] ORIGINAL : {row['text'][:120]}")
    print(f"    CLEANED  : {row['cleaned_text'][:120]}")

---
## Step 4 — AI Sentiment & Topic Analysis (Gemini 1.5 Flash)

Each cleaned tweet is sent to **Google Gemini 1.5 Flash** with a structured prompt.
The model returns a JSON object with:
- `sentiment`: one of **Positive**, **Negative**, or **Neutral**
- `topic`: a short 1–3 word phrase summarising the tweet's subject

> **API Key required.** Add `GEMINI_API_KEY` to your Colab Secrets (free tier available at [aistudio.google.com](https://aistudio.google.com/app/apikey)).

In [ ]:
import json
import time
import google.generativeai as genai

# ── Configure Gemini API ──────────────────────────────────────────────────────
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    raise EnvironmentError(
        "GEMINI_API_KEY not found.\n"
        "Please add it to Colab Secrets (🔑 icon in the left sidebar) "
        "or set it as an environment variable.\n"
        "Get a free key at: https://aistudio.google.com/app/apikey"
    )

genai.configure(api_key=GEMINI_API_KEY)
print("✅ Gemini API configured successfully.")

# ── Load cleaned data ─────────────────────────────────────────────────────────
df_analysis = pd.read_csv("cleaned_tweets.csv")
# Analyse first 20 tweets to stay well within free-tier quota
df_analysis = df_analysis.head(20).copy()
print(f"✅ Loaded {len(df_analysis)} cleaned tweets for analysis.")

# ── Gemini model setup ────────────────────────────────────────────────────────
model = genai.GenerativeModel('gemini-1.5-flash')

PROMPT_TEMPLATE = """\
Analyze the sentiment and the main topic of the following tweet.
The sentiment must be exactly one of: Positive, Negative, or Neutral.
The topic should be a short, 1-3 word phrase describing the subject.
Return ONLY a valid JSON object with exactly two keys: "sentiment" and "topic".

Tweet: "{tweet_text}"

JSON Output:"""


def analyze_tweet(tweet_text: str) -> dict | None:
    """Send a single tweet to Gemini and parse the JSON sentiment/topic response."""
    if not tweet_text or not tweet_text.strip():
        return {"sentiment": "Neutral", "topic": "N/A"}

    prompt = PROMPT_TEMPLATE.format(tweet_text=tweet_text)

    try:
        response = model.generate_content(prompt)
        raw = response.text.strip()

        # Strip markdown code fences if present (e.g. ```json ... ```)
        raw = re.sub(r'^```(?:json)?\s*', '', raw, flags=re.MULTILINE)
        raw = re.sub(r'```\s*$', '', raw, flags=re.MULTILINE)
        raw = raw.strip()

        result = json.loads(raw)

        # Normalise sentiment to expected values
        sentiment = result.get("sentiment", "Neutral").strip().capitalize()
        if sentiment not in ("Positive", "Negative", "Neutral"):
            sentiment = "Neutral"

        return {"sentiment": sentiment, "topic": result.get("topic", "General").strip()}

    except json.JSONDecodeError:
        print(f"  ⚠️  Could not parse JSON response: {raw[:80]}")
        return {"sentiment": "Error", "topic": "Error"}
    except Exception as e:
        print(f"  ⚠️  API error: {e}")
        return {"sentiment": "Error", "topic": "Error"}


# ── Run analysis ──────────────────────────────────────────────────────────────
print("\n🔄 Analyzing tweets with Gemini 1.5 Flash...")
results = []

for idx, row in df_analysis.iterrows():
    print(f"  Processing tweet {idx + 1}/{len(df_analysis)}...", end="\r")
    result = analyze_tweet(row['cleaned_text'])
    results.append(result)
    time.sleep(0.5)  # respect API rate limits

print("\n✅ Analysis complete!")

# ── Merge results ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results)
df_analysis['sentiment'] = results_df['sentiment'].values
df_analysis['topic'] = results_df['topic'].values

df_analysis.to_csv("analyzed_tweets.csv", index=False)
print("✅ Results saved to 'analyzed_tweets.csv'.")

print("\n--- Sample Results ---")
print(df_analysis[['text', 'sentiment', 'topic']].head(5).to_string())

---
## Step 5 — Results Visualization

Generates two charts:
1. **Sentiment Distribution** — how many tweets are Positive, Negative, Neutral
2. **Top Topics** — the most frequently mentioned subjects across all tweets

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Load analyzed data ────────────────────────────────────────────────────────
try:
    df_viz = pd.read_csv("analyzed_tweets.csv")
    # Filter out error rows for clean visualisation
    df_viz = df_viz[df_viz['sentiment'].isin(['Positive', 'Negative', 'Neutral'])]
    print(f"✅ Loaded {len(df_viz)} analyzed tweets for visualization.")
except FileNotFoundError:
    print("❌ 'analyzed_tweets.csv' not found. Please run Step 4 first.")
    raise

# ── Colour palette ────────────────────────────────────────────────────────────
SENTIMENT_COLORS = {
    'Positive': '#2ECC71',
    'Neutral':  '#3498DB',
    'Negative': '#E74C3C',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    'Social Media Sentiment & Topic Analysis — Electric Vehicles',
    fontsize=15, fontweight='bold', y=1.02
)

# ── Chart 1: Sentiment Distribution (Bar) ────────────────────────────────────
ax1 = axes[0]
sentiment_counts = df_viz['sentiment'].value_counts().reindex(
    ['Positive', 'Neutral', 'Negative'], fill_value=0
)
bars = ax1.bar(
    sentiment_counts.index,
    sentiment_counts.values,
    color=[SENTIMENT_COLORS[s] for s in sentiment_counts.index],
    edgecolor='white',
    linewidth=1.5,
    width=0.55
)

# Add count labels on bars
for bar in bars:
    height = bar.get_height()
    ax1.text(
        bar.get_x() + bar.get_width() / 2.,
        height + 0.1,
        f'{int(height)}',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )

ax1.set_title('Sentiment Distribution', fontsize=13, fontweight='bold', pad=12)
ax1.set_xlabel('Sentiment', fontsize=11)
ax1.set_ylabel('Number of Tweets', fontsize=11)
ax1.set_ylim(0, sentiment_counts.max() + 2)
ax1.spines[['top', 'right']].set_visible(False)
ax1.yaxis.grid(True, linestyle='--', alpha=0.5)
ax1.set_axisbelow(True)

# ── Chart 2: Top 10 Topics (Horizontal Bar) ───────────────────────────────────
ax2 = axes[1]
topic_counts = (
    df_viz['topic']
    .str.lower()
    .str.strip()
    .value_counts()
    .head(10)
    .iloc[::-1]  # reverse for horizontal readability
)

colors = plt.cm.Blues([0.4 + 0.06 * i for i in range(len(topic_counts))])
h_bars = ax2.barh(
    topic_counts.index,
    topic_counts.values,
    color=colors,
    edgecolor='white',
    linewidth=1.2
)

for bar in h_bars:
    width = bar.get_width()
    ax2.text(
        width + 0.05, bar.get_y() + bar.get_height() / 2,
        f'{int(width)}',
        va='center', ha='left', fontsize=10
    )

ax2.set_title('Top 10 Discussed Topics', fontsize=13, fontweight='bold', pad=12)
ax2.set_xlabel('Frequency', fontsize=11)
ax2.set_xlim(0, topic_counts.max() + 1)
ax2.spines[['top', 'right']].set_visible(False)
ax2.xaxis.grid(True, linestyle='--', alpha=0.5)
ax2.set_axisbelow(True)

plt.tight_layout()
plt.savefig('sentiment_analysis_results.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary statistics ────────────────────────────────────────────────────────
total = len(df_viz)
print("\n📊 Summary Statistics")
print("-" * 35)
for sentiment, count in sentiment_counts.items():
    pct = count / total * 100 if total else 0
    print(f"  {sentiment:<12}: {count:>3} tweets ({pct:.1f}%)")
print("-" * 35)
print(f"  Total analyzed: {total} tweets")
print(f"\n📁 Chart saved to 'sentiment_analysis_results.png'")